# Knee MRI: leave-one-vendor-out test (Philips held out)
Trains the baseline recipe (old labels, EfficientNet-B0, 160 px, 12 epochs) on GE + Siemens + Toshiba and scores it on **all Philips studies**.
Compares with the baseline's site-grouped OOF predictions on the same studies. Two seeds (42 and 7) run in parallel, one per GPU, ~1 h wall.

**Settings:** GPU T4 x2, internet ON (`--pretrained`), *Save & Run All*.
**Inputs:** `knee-mri-pipeline` dataset (unmodified; `train.py` is patched at runtime), caching notebook output (cache + `labels_soft.csv`), baseline training output (`oof_all.csv`).

In [1]:
import os, sys, glob, json, shutil, subprocess, time
import numpy as np, pandas as pd

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
KEY4 = LABELS[:4]                      # ACL, MCL, Medial Meniscus, Lateral Meniscus
W = "/kaggle/working"

def launch(gpu, out, extra, code, train_args):
    """Start one train.py process pinned to one GPU. Returns (Popen, logfile) or None if already running."""
    name = os.path.basename(out)
    log = f"{W}/{name}.log"
    if subprocess.run(["pgrep", "-f", "--", f"--out {out}"], capture_output=True).returncode == 0:
        print(f"[{name}] already running -> skipped (launch guard)")
        return None
    cmd = [sys.executable, f"{code}/train.py", *train_args, "--out", out, *extra]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    print(f"[{name}] GPU {gpu}:", " ".join(cmd))
    return subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT), log

def last_match(log, pat):
    try:
        lines = [l.rstrip() for l in open(log, errors="ignore") if pat in l]
        return lines[-1] if lines else "(none yet)"
    except FileNotFoundError:
        return "(no log)"

def wait_all(jobs, every=900):
    """Block until every process ends (REQUIRED in a committed notebook: it kills children when the last cell ends)."""
    t0, last = time.time(), 0.0
    while any(p.poll() is None for p, _ in jobs.values()):
        time.sleep(20)
        if time.time() - last >= every:
            last = time.time()
            print(f"[{(time.time() - t0) / 3600:.2f} h]", {k: last_match(l, " ep ") for k, (_, l) in jobs.items()}, flush=True)
    bad = []
    for k, (p, l) in jobs.items():
        print(f"{k} exit code {p.returncode}")
        if p.returncode != 0:
            bad.append(k); print("---- tail of", l); print("".join(open(l, errors="ignore").readlines()[-25:]))
    return bad

CODE_SRC = "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"
CACHE    = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"
BASE     = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"
LABELS_OLD_OVERRIDE = None     # path of the ORIGINAL labels_soft.csv if the search below finds 0 or 2 files
CODE = f"{W}/knee_pipeline"

for p in (CODE_SRC, CACHE, BASE):
    assert os.path.exists(p), f"missing input: {p}"
cands = [LABELS_OLD_OVERRIDE] if LABELS_OLD_OVERRIDE else [
    os.path.join(os.path.dirname(CACHE), "labels_soft.csv"), os.path.join(CACHE, "labels_soft.csv")]
cands = [c for c in cands if c and os.path.exists(c)]
assert len(cands) == 1, f"original labels_soft.csv: {len(cands)} candidates {cands}. ls the caching notebook output and set LABELS_OLD_OVERRIDE."
LABELS_OLD = cands[0]
print("labels:", LABELS_OLD)

lab, base = pd.read_csv(LABELS_OLD), pd.read_csv(BASE)
need = ["StudyInstanceUID", "is_gold"] + [f"{p}_{l}" for p in "yw" for l in LABELS]
assert not [c for c in need if c not in lab.columns], [c for c in need if c not in lab.columns]
assert not [c for c in ["StudyInstanceUID"] + [f"{p}_{l}" for p in "py" for l in LABELS] if c not in base.columns], "oof_all.csv lacks p_/y_ columns"
m = lab.merge(base[["StudyInstanceUID"] + [f"y_{l}" for l in LABELS]], on="StudyInstanceUID", suffixes=("", "_b"))
d = max(np.nanmax(np.abs(m[f"y_{l}"] - m[f"y_{l}_b"])) for l in LABELS)
print(f"label rows {len(lab)} | baseline rows {len(base)} | merged {len(m)} | max |y_ labels_soft - y_ oof_all| = {d:.4f}")
assert len(m) == 4407 and d < 1e-3, "labels_soft.csv is not the file the baseline was trained on: STOP"
print("setup OK")

labels: /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv
label rows 4407 | baseline rows 4407 | merged 4407 | max |y_ labels_soft - y_ oof_all| = 0.0000
setup OK


In [2]:
# ---- copy the code and add --fold_file to train.py (exact-match patch; fails loudly if train.py differs)
if os.path.exists(CODE): shutil.rmtree(CODE)
shutil.copytree(CODE_SRC, CODE)
tp = f"{CODE}/train.py"; s = open(tp).read()

def sub(old, new):
    global s
    assert s.count(old) == 1, f"patch anchor not found exactly once: {old.strip()[:70]!r}"
    s = s.replace(old, new)

if "--fold_file" in s:
    print("train.py already has --fold_file; no patch applied")
else:
    sub('    ap.add_argument("--bootstrap", type=int, default=500)\n',
        '    ap.add_argument("--bootstrap", type=int, default=500)\n'
        '    ap.add_argument("--fold_file", default=None, help="CSV StudyInstanceUID,fold (0..n-1) overriding GroupKFold")\n')
    sub('    if groups.nunique() >= a.folds:\n',
        """    if a.fold_file:
        ff = pd.read_csv(a.fold_file)[["StudyInstanceUID", "fold"]].drop_duplicates("StudyInstanceUID")
        fv = df[["StudyInstanceUID"]].merge(ff, on="StudyInstanceUID", how="left")["fold"]
        if fv.isna().any():
            raise ValueError(f"{int(fv.isna().sum())} studies are missing from --fold_file")
        fv = fv.astype(int).values
        ks = sorted(set(fv.tolist()))
        if ks != list(range(len(ks))):
            raise ValueError(f"--fold_file folds must be 0..n-1, got {ks}")
        splits = [(np.where(fv != k)[0], np.where(fv == k)[0]) for k in ks]
        print(f"fold_file: {a.fold_file} | val sizes per fold {[len(sp[1]) for sp in splits]}")
    elif groups.nunique() >= a.folds:
""")
    sub('    check_folds(df)\n', '    if not a.fold_file:\n        check_folds(df)\n')
    open(tp, "w").write(s)
    print("patched", tp)
import py_compile; py_compile.compile(tp, doraise=True); print("compiles OK")

patched /kaggle/working/knee_pipeline/train.py
compiles OK


In [3]:
# ---- fold file: fold 0 = Philips (validation), fold 1 = everything else (training)
idx = pd.read_csv(f"{CACHE}/cache_index.csv")
assert (idx.ok == 1).all() and len(idx) == 4407, (len(idx), int((idx.ok != 1).sum()))

def vendor(sp):
    m = str(sp).split("|")[0].strip().upper()
    if "PHILIPS" in m: return "Philips"
    if "SIEMENS" in m: return "Siemens"
    if "TOSHIBA" in m or "CANON" in m: return "Toshiba/Canon"
    if m.startswith("GE") or "GENERAL" in m: return "GE"
    return "Other"

idx["vendor"] = idx.site_proxy.map(vendor)
print("raw manufacturer prefixes (top 15):"); print(idx.site_proxy.str.split("|").str[0].value_counts().head(15).to_string())
print("\nvendor counts:", idx.vendor.value_counts().to_dict(), "(spec expects ~ GE 868, Siemens 1951, Philips 1301, Toshiba 181)")
n_ph = int((idx.vendor == "Philips").sum())
assert 1100 <= n_ph <= 1500, f"Philips count {n_ph} is off: fix vendor() before training"
print("Other vendor studies (kept in training):", int((idx.vendor == "Other").sum()))
FOLDS = f"{W}/folds_philips.csv"
idx.assign(fold=(idx.vendor != "Philips").astype(int))[["StudyInstanceUID", "fold"]].to_csv(FOLDS, index=False)
print(f"saved {FOLDS} | Philips (val) {n_ph} | train {len(idx) - n_ph}")

raw manufacturer prefixes (top 15):
site_proxy
Siemens Heal    1053
GE MEDICAL S     868
SIEMENS          804
Philips Medi     718
Philips          492
TOSHIBA          181
Siemens           94
Philips Heal      91
CANON_MEC         45
GEHC              37
FUJIFILM Hea      16
Hitachi Medi       8

vendor counts: {'Siemens': 1951, 'Philips': 1301, 'GE': 905, 'Toshiba/Canon': 226, 'Other': 24} (spec expects ~ GE 868, Siemens 1951, Philips 1301, Toshiba 181)
Other vendor studies (kept in training): 24
saved /kaggle/working/folds_philips.csv | Philips (val) 1301 | train 3106


In [4]:
# ---- two seeds in parallel, one per GPU
TRAIN_ARGS = ["--cache_dir", CACHE, "--labels", LABELS_OLD, "--fold_file", FOLDS, "--only_folds", "0",
              "--epochs", "12", "--pretrained", "--workers", "2", "--train_slices", "16", "--bs", "4", "--accum", "2"]
jobs = {}
for gpu, seed in ((0, 42), (1, 7)):
    r = launch(gpu, f"{W}/philips_s{seed}", ["--seed", str(seed)], CODE, TRAIN_ARGS)
    if r: jobs[f"s{seed}"] = r
time.sleep(150)
for k, (p, l) in jobs.items():
    print(k, "|", last_match(l, "fold_file:"), "|", last_match(l, "===== fold"))
bad = wait_all(jobs)
assert not bad, f"failed: {bad}"

[philips_s42] GPU 0: /usr/bin/python3 /kaggle/working/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --fold_file /kaggle/working/folds_philips.csv --only_folds 0 --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --out /kaggle/working/philips_s42 --seed 42
[philips_s7] GPU 1: /usr/bin/python3 /kaggle/working/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --fold_file /kaggle/working/folds_philips.csv --only_folds 0 --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --out /kaggle/working/philips_s7 --seed 7
s42 | fold_file: /kaggle/working/folds_philips.csv | val sizes per fold [1301, 3106] | ===== fold 0 | train 3106 val 1301 =====
s7 | fold_file: /kaggle/working/folds

In [5]:
from sklearn.metrics import roc_auc_score

def aucs(Y, P, rows=None):
    """Per-label AUC on CONFIDENT old labels (y>=0.9 positive, y<=0.1 negative). Y, P: [N,12]."""
    out = []
    for j in range(12):
        y = Y[:, j]; m = (y >= 0.9) | (y <= 0.1)
        if rows is not None: m = m & rows
        t = (y[m] >= 0.9).astype(int)
        out.append(roc_auc_score(t, P[m, j]) if 0 < t.sum() < len(t) else np.nan)
    return np.array(out)

summ = lambda a: (float(np.nanmean(a)), float(np.nanmean(a[:4])))     # (macro, KEY-4)

runs = {s: pd.read_csv(f"{W}/philips_{s}/oof.csv").set_index("StudyInstanceUID") for s in ("s42", "s7")}
pc, yc = [f"p_{l}" for l in LABELS], [f"y_{l}" for l in LABELS]
ids = runs["s42"].index[runs["s42"].p_ACL.notna()]
assert set(ids) == set(runs["s7"].index[runs["s7"].p_ACL.notna()]), "seeds scored different studies"
b = base.set_index("StudyInstanceUID").loc[ids]
Y, Pb = b[yc].values, b[pc].values
Ps = {s: r.loc[ids, pc].values for s, r in runs.items()}
print(f"Philips studies scored: {len(ids)} | max |y_ run - y_ baseline| = {np.nanmax(np.abs(runs['s42'].loc[ids, yc].values - Y)):.4f}")

A = pd.DataFrame({"baseline (site-grouped OOF)": aucs(Y, Pb), "held-out s42": aucs(Y, Ps["s42"]), "held-out s7": aucs(Y, Ps["s7"])}, index=LABELS)
A["delta (mean of seeds - baseline)"] = A[["held-out s42", "held-out s7"]].mean(1) - A.iloc[:, 0]
print(A.round(3).to_string())
S = {c: summ(A[c].values) for c in A.columns[:3]}
print("\n            macro   KEY-4"); [print(f"{c:28s} {v[0]:.4f}  {v[1]:.4f}") for c, v in S.items()]
mb = S["baseline (site-grouped OOF)"]
dm = [(S[f"held-out {s}"][0] - mb[0], S[f"held-out {s}"][1] - mb[1]) for s in ("s42", "s7")]
print(f"\nmacro delta per seed: {dm[0][0]:+.4f} / {dm[1][0]:+.4f} | KEY-4 delta: {dm[0][1]:+.4f} / {dm[1][1]:+.4f} | seed-to-seed macro gap {abs(S['held-out s42'][0] - S['held-out s7'][0]):.4f}")
if not 0.73 <= mb[0] <= 0.77: print("WARNING: baseline Philips macro is far from the ~0.75 in the spec; check labels/ids before reading anything")

rng, n, ci = np.random.default_rng(0), len(ids), {}
for s in ("s42", "s7"):
    d = []
    for _ in range(300):
        i = rng.integers(0, n, n); a, c = summ(aucs(Y[i], Ps[s][i])), summ(aucs(Y[i], Pb[i])); d.append((a[0] - c[0], a[1] - c[1]))
    ci[s] = np.percentile(np.array(d), [5, 95], axis=0)
    print(f"{s}: macro delta 90% CI [{ci[s][0, 0]:+.3f}, {ci[s][1, 0]:+.3f}] | KEY-4 delta 90% CI [{ci[s][0, 1]:+.3f}, {ci[s][1, 1]:+.3f}]  (study resampling only)")

g = float(np.mean([dm[0][0], dm[1][0]]))
print("\nVERDICT (my rough thresholds, not a test):")
print("  macro drop > 0.03 : real vendor shift -> worth targeted work (augmentation, harmonisation)." if g < -0.03 else
      "  macro drop 0.015-0.03: some vendor shift; weigh against the ~12% smaller training set (confound)." if g < -0.015 else
      "  macro drop < 0.015: no detectable vendor shift on this check; keep ensembling, skip vendor-specific work.")
print("  Caveat: the baseline for these studies saw other Philips sites in training; the held-out model saw none, and had ~3,100 vs ~3,500 training studies.")

Philips studies scored: 1301 | max |y_ run - y_ baseline| = 0.0000
                  baseline (site-grouped OOF)  held-out s42  held-out s7  delta (mean of seeds - baseline)
ACL                                     0.757         0.726        0.743                            -0.023
MCL                                     0.706         0.685        0.690                            -0.019
Medial Meniscus                         0.731         0.728        0.732                            -0.001
Lateral Meniscus                        0.731         0.736        0.740                             0.007
Medial OA                               0.853         0.845        0.854                            -0.003
Lateral OA                              0.762         0.758        0.766                            -0.000
PF OA                                   0.713         0.712        0.717                             0.002
Effusion                                0.826         0.835        0.835     